# 🧽 Remover Legenda — ProPainter (memória-segura, vídeo longo)
Grátis na **GPU do Colab**. Remove legenda de **qualquer cor/posição**, processando só a
**tira da legenda** em **pedaços** (não derruba a sessão).

**Passos:** 1) *Ambiente de execução → Alterar tipo → GPU (T4)*  2) *Executar tudo*  3) envie o vídeo.

> No 1º teste deixe `MAX_SEGUNDOS = 15` (rápido). Funcionando, ponha `0` (vídeo inteiro) e rode de novo.

In [ ]:
# 1) GPU
!nvidia-smi -L || echo 'SEM GPU: Ambiente de execucao > Alterar tipo > GPU (T4)'

In [ ]:
# 2) ProPainter oficial + pesos
import os
if not os.path.isdir('/content/ProPainter'):
    !cd /content && git clone -q https://github.com/sczhou/ProPainter.git
%cd /content/ProPainter
!pip install -q -r requirements.txt
os.makedirs('weights', exist_ok=True)
base='https://github.com/sczhou/ProPainter/releases/download/v0.1.0'
for w in ['ProPainter.pth','raft-things.pth','recurrent_flow_completion.pth']:
    if not os.path.exists('weights/'+w) or os.path.getsize('weights/'+w)<100000:
        !wget -q -O weights/{w} {base}/{w}
print('ProPainter ok:', [(w,os.path.getsize('weights/'+w)) for w in os.listdir('weights')])

In [ ]:
# 3) Detector de texto (ONNX)
!pip install -q rapidocr-onnxruntime
print('ok')

In [ ]:
# 4) Envie o video
from google.colab import files
up=files.upload(); VIDEO=list(up.keys())[0]; print('video:', VIDEO)

In [ ]:
# 5) Extrai quadros p/ DISCO, acha a TIRA da legenda e faz as mascaras (pouca RAM)
MAX_SEGUNDOS = 15   # <<< 15 = teste rapido. Ponha 0 p/ o video INTEIRO.
DILATE = 12; MIN_AREA = 60
import os, cv2, glob, shutil, numpy as np, subprocess
from rapidocr_onnxruntime import RapidOCR
ocr = RapidOCR()
_rate = subprocess.check_output(['ffprobe','-v','0','-of','csv=p=0','-select_streams','v:0',
        '-show_entries','stream=r_frame_rate', VIDEO]).decode().strip()
_n,_d = (_rate.split('/')+['1'])[:2]; fps = float(_n)/float(_d or 1)
if os.path.isdir('frames_all'): shutil.rmtree('frames_all')
os.makedirs('frames_all')
lim = ['-t', str(MAX_SEGUNDOS)] if MAX_SEGUNDOS>0 else []
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',VIDEO]+lim+['frames_all/%05d.png'], check=True)
ff = sorted(glob.glob('frames_all/*.png')); N=len(ff)
H,W = cv2.imread(ff[0]).shape[:2]
print(f'{N} quadros {W}x{H} @ {fps:.1f}')

def boxes(img):
    out=ocr(img,use_det=True,use_rec=False,use_cls=False); res=out[0] if isinstance(out,tuple) else out
    bs=[]
    if res:
        for it in res:
            a=np.asarray(it,dtype=np.float32); b=a if (a.ndim==2 and a.shape==(4,2)) else np.asarray(it[0],dtype=np.float32)
            if b.shape==(4,2) and cv2.contourArea(b)>=MIN_AREA: bs.append(b)
    return bs

# acha a TIRA amostrando ~50 quadros
uni=np.zeros((H,W),np.uint8)
for i in np.linspace(0,N-1,min(N,50)).astype(int):
    for b in boxes(cv2.imread(ff[i])): cv2.fillPoly(uni,[b.astype(np.int32)],255)
ys=np.where(uni.max(axis=1)>0)[0]
if len(ys)==0: raise SystemExit('Nenhum texto detectado.')
Y0=max(0,int(ys.min())-24); Y1=min(H,int(ys.max())+24)
if (Y1-Y0)%2: Y1=min(H,Y1+1)
print('tira da legenda: y', Y0,'-',Y1,' (altura',Y1-Y0,'de',H,')')

for d in ['band_f','band_m']:
    if os.path.isdir(d): shutil.rmtree(d)
    os.makedirs(d)
k=max(1,DILATE); ker=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k*2+1,k*2+1))
hits=0
for i in range(N):
    fr=cv2.imread(ff[i]); crop=fr[Y0:Y1]
    m=np.zeros((Y1-Y0,W),np.uint8)
    for b in boxes(crop): cv2.fillPoly(m,[b.astype(np.int32)],255)
    if m.any(): m=cv2.dilate(m,ker); hits+=1
    cv2.imwrite(f'band_f/{i:05d}.png', crop)
    cv2.imwrite(f'band_m/{i:05d}.png', m)
    if (i+1)%100==0 or i==N-1: print(f'  mascaras {i+1}/{N}')
print(f'pronto: {N} quadros da tira, {hits} com texto')
if hits==0: raise SystemExit('Nenhum texto na tira.')

In [ ]:
# 5b) LIMPA falsos-positivos do ROSTO por PERSISTENCIA e recorta a faixa justa
#     (texto real fica FIXO em quase todo quadro; erro no rosto e' intermitente)
import glob, cv2, numpy as np
from google.colab.patches import cv2_imshow
ps = sorted(glob.glob('band_m/*.png')); ff2 = sorted(glob.glob('frames_all/*.png'))
acc = np.zeros(cv2.imread(ps[0],0).shape, np.float32)
for p in ps: acc = acc + (cv2.imread(p,0) > 0)
acc = acc / len(ps)
fx = (acc >= 0.5).astype(np.uint8) * 255            # aparece em >=50% dos quadros = texto fixo
fx = cv2.dilate(fx, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)))
ry = np.where(fx.max(1) > 0)[0]
if len(ry) == 0: raise SystemExit('Nenhum texto FIXO encontrado (baixe o 0.5).')
g0 = Y0 + int(ry.min()); g1 = Y0 + int(ry.max()) + 1
nY0 = max(0, g0 - 8); nY1 = min(H, g1 + 8); nY1 = nY1 - ((nY1 - nY0) % 2)
top = g0 - nY0
full = np.zeros((nY1 - nY0, fx.shape[1]), np.uint8)
full[top:top + (g1 - g0)] = fx[int(ry.min()):int(ry.max()) + 1]
for i in range(len(ps)):
    cv2.imwrite('band_f/%05d.png' % i, cv2.imread(ff2[i])[nY0:nY1])
    cv2.imwrite('band_m/%05d.png' % i, full)
Y0, Y1 = nY0, nY1
prev = cv2.imread('band_f/%05d.png' % (len(ps) // 2)).copy(); prev[full > 0] = [0, 0, 255]
print('faixa ajustada: y', Y0, Y1, 'altura', Y1 - Y0, 'texto_px', int((full > 0).sum()))
print('VERMELHO = sera apagado; o ROSTO deve ficar LIMPO'); cv2_imshow(prev)

In [ ]:
# 6) ProPainter em PEDACOS (subprocess por pedaco = RAM liberada entre eles)
import os, cv2, glob, shutil, subprocess
SEG = 240   # ~8s por pedaco
bf=sorted(glob.glob('band_f/*.png')); bm=sorted(glob.glob('band_m/*.png')); N=len(bf)
if os.path.isdir('out_band'): shutil.rmtree('out_band')
os.makedirs('out_band')
for s in range(0,N,SEG):
    e=min(s+SEG,N)
    for d in ['seg_f','seg_m','results']:
        if os.path.isdir(d): shutil.rmtree(d)
    os.makedirs('seg_f'); os.makedirs('seg_m')
    for j,i in enumerate(range(s,e)):
        shutil.copy(bf[i],f'seg_f/{j:05d}.png'); shutil.copy(bm[i],f'seg_m/{j:05d}.png')
    print(f'pedaco {s}-{e} de {N} ...', flush=True)
    subprocess.run(['python','inference_propainter.py','--video','seg_f','--mask','seg_m',
        '--save_fps','30','--mask_dilation','0','--fp16','--subvideo_length','80',
        '--neighbor_length','10','--ref_stride','10'], check=True)
    outv=sorted(glob.glob('results/*/inpaint_out.mp4'))[-1]
    cap=cv2.VideoCapture(outv); j=0
    while True:
        ok,fr=cap.read()
        if not ok: break
        cv2.imwrite(f'out_band/{s+j:05d}.png',fr); j+=1
    cap.release(); print(f'  ok ({j} quadros)')
print('todos os pedacos prontos:', len(os.listdir('out_band')))

In [ ]:
# 7) Cola a tira limpa no quadro, devolve o audio e baixa
import cv2, glob, os, shutil
ff=sorted(glob.glob('frames_all/*.png')); ob=sorted(glob.glob('out_band/*.png')); N=len(ob)
if os.path.isdir('final_frames'): shutil.rmtree('final_frames')
os.makedirs('final_frames')
for i in range(N):
    full=cv2.imread(ff[i]); band=cv2.imread(ob[i])
    full[Y0:Y1]=cv2.resize(band,(full.shape[1],Y1-Y0))
    cv2.imwrite(f'final_frames/{i:05d}.png', full)
SAIDA='video_sem_legenda.mp4'
os.system(f'ffmpeg -y -loglevel error -framerate {fps} -i final_frames/%05d.png -i "{VIDEO}" '
          f'-map 0:v -map 1:a? -c:v libx264 -crf 18 -pix_fmt yuv420p -c:a aac -shortest "{SAIDA}"')
print('PRONTO ->', SAIDA, os.path.getsize(SAIDA),'bytes')
from google.colab import files; files.download(SAIDA)